In [1]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

from sklearn.metrics import classification_report

from peft import LoraConfig, get_peft_model

/Users/pengu/Developer/Deep Learning PyTorch Codebook/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'sklearn'

# Finetuning
pretrained model + task data + task head/objective -> adapted model
- Compose of tokenizer + model + data + loss + metrics
## Workflow
1. Choose pretrained ckpt, match tokenizer & processor
2. Define task formulation and output format
3. Build dataset, split, collate function, padding/truncation logic
4. Add/replace task head
5. Choose optimization recipe
6. Train, validate, save best ckpt
7. Evaluate using task-relevant metrics
8. Export/deploy with the same preprocessing contract.
RULE: ckpt, tokenizer, preprocessing, head must be consistent
## Finetuning mode
- Linear probing: train only task head
+ Fast, stable, low memory usage
+ Limited adaptation
- Full finetuning: train all parameters
+ Strongest adaptation
+ Expensive, overfit risk
- Partial unfreezing: train top layers + head
+ Balanced
+ Layer choice matters
- Parameter Efficient Fine-tuning (PEFT)/Low-rank Adaption (LoRA): train small adapter params
+ Memory-efficient
+ Extra design choices
- Default choice: pretrained model + correct tokenizer + simple full fine-tuning/LoRA
## Task - Head Harmony
- Classification: Class logits -> linear head + CrossEntropyLoss
- Regression: Scalar/vector -> linear head + MSE/MAE loss
- Token classification: Logits per token -> linear head + token CE loss
- Question answering: start/end logits -> two linear heads
- Causal LM: vocab logits per step -> LM head + CE loss
- Seq2seq: target token sequence -> decoder LM loss
- Guide: Choose target tensor & loss first, then model API
## Input-output shape contract
### Text classification
- input_ids, attention_mask (B, n)
- logits (B, C)
### Token classification
- logits (B, n, C) / (B, n)

In [ ]:
ckpt = 'bert-base-uncased'
tok = AutoTokenizer.from_pretrained(ckpt)

model = AutoModelForSequenceClassification.from_pretrained(
    ckpt, 
    num_labels=2
)

batch = tok(
    ['A good movie.', 'A bad movie.'],
    padding=True,
    truncation=True,
    return_tensors='pt',
)
outputs = model(**batch)
print(outputs.logits)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11612.11it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

tensor([[ 0.0187, -0.0671],
        [ 0.0673, -0.1142]], grad_fn=<AddmmBackward0>)


## Checkpoint-tokenizer contract
- Must match: Vocabulary, special tokens, padding side, max sequence length, preprocessing convention
- Mismatching result in silently degrade performance
## Dataset contract
- Sample consists of:
+ raw text/tokenized ids
+ label/target
+ metadata (for later analysis)
- Keep raw data and labels clean, apply tokenizer consistently in whole processing and dataset mapping
## Padding strategies
- Static padding: pad all samples to global max length (simple but wasteful)
- Dynamic padding: pad to max length of each batch (efficient default)
- Bucketing: group similar lengths together (improves efficency further, but take times)
- Packing: combine short sequences (useful for LM pretraining/finetuning, take more times)
- Affects both memory and speed

In [ ]:
# Dynamic padding collator
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(
    tokenizer=tok,
    padding=True,
    return_tensors='pt',
)

In [ ]:
# Dataset mapping to dynamic padding collator
def tokenize_batch(batch):
    return tok(
        batch['text'],
        padding=True,
        truncation=True,
        return_tensors='pt',
    )

tokenized_ds = raw_ds.map(
    tokenize_batch, 
    batched=True
    remove_columns=['text']
)
# Tokenize once if dataset is stable, tokenize on the fly only if necessary

## Truncation
- Max length is too short -> importance evidence may be removed
- Increase max length if hardware allows
- Use sliding window for long documents
- Use hierarchical pooling over chunks
- Use long-context architectures when needed
- Truncate can become hidden noise
## Optimization
### Checklist example
- Optimizer: AdamW
- Learning rate: warmup + decay
- Regularization: dropout, weight decay, label smoothing
- Stability: gradient clipping
- Memory/speed: mixed precision
- Model selection: save best ckpt by validation metric
- IMPORTANT: Finetuning is sensitive to these optimization details
### Learning rate
- Practical ranges
+ Full finetuning BERT-like: 1e-5 -> 5e-5
+ Training only classifier head: 1e-4 -> 1e-3
+ LoRA/adapter tuning: 1e-4 -> 3e-4
+ Larger decoder finetuning: conservative, task/hardware dependent
- Large pretrained models need smaller LR than models trained from scratch
### Warmup and decay
- Warmup: start with small LR and increase gradually n_t = n_max(t/T_warmup)
- Decay: after warmup, reduce LR using linear/cosine/inverse-square-root schedule
- Warmup prevents early destructive updates, decay improves final convergence

In [ ]:
# AdamW + linear warmup + linear decay
optim = torch.optim.AdamW(
    model.parameters(), 
    lr=2e-5,
    weight_decay=0.01,
)

num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optim,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

In [ ]:
# Training loop
scaler = torch.amp.GradScaler()

model.train()

for batch in train_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}

    optim.zero_grad(set_to_none=True)

    with torch.amp.autocast():
        outputs = model(**batch)
        loss = outputs.loss
    
    scaler.scale(loss).backward()
    scaler.unscale_(optim)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    scaler.step(optim)
    scaler.update()
    scheduler.step()

## Freezing and gradual unfreezing
Freezing controls adaptation capacity and overfitting risk
- IMPORTANT: print the trainable parameter count before training
### Head-only training
Freeze backbone, train only task head -> quick baseline
### Full finetuning
Unfreeze all parameters -> stronger baseline, more expensive
### Gradual unfreezing
Unfreeze top layers first, then lower layers if needed

In [ ]:
# Freezing backbone example
for name, p in model.named_parameters():
    p.requires_grad = False

# Unfreeze classification head
for name, p in model.classifier.named_parameters():
    p.requires_grad = True

## Evaluation and metrics
Metrics use to evaluate the output
### Task - metrics harmony
- Classfication: accuracy, macro-F1, weighted-F1
- Imbalance classification: macro-F1, per-class recall
- NER/token classification: entity-level F1
- QA: exact match, F1
- Generation: ROUGE, BLEU, BERTScore, human evaluation
- Retrieval: Recall@K, Mean Reciprocal Rank (MRR), Normalized Discounted Cumulative Gain (NDCG)
### Validation discipline
- Use validation set for hyperparameters (model selection, ...)
- Use test set only for final report
- Save best ckpt by validation metrics
- Track confusion matrix and error examples
- Use same preprocessing at every step (train, validation, test, deployment)
- IMPORTANT: Do not tune hyperparameters on the test set

In [ ]:
# Metric computation sketch
model.eval()
all_preds, all_golds = [], []

with torch.no_grad():
    for batch in val_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        preds = outputs.logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_golds.extend(batch['labels'].cpu().tolist())

print(classification_report(all_golds, all_preds))

## Parameter-efficient finetuning (PEFT)
### Why PEFT
- Full finetuning LM is expensive:
+ many trainable parameters
+ high optimizer memory
+ difficult multi-task storage
+ higher risk of overfitting on small datasets
- PEFT freeze most pretrained weights, train small numbers of extra/low rank parameters
- Make large model adaptation cheaper
### Methods
- Adapters: insert small bottleneck modules -> modular, extra latency
- Prefix tuning: learn prefix key/val vector -> useful for generation
- Prompt tuning: learn soft prompt embeddings -> very parameter-efficient
- LoRA: low-rank updates to linear layers -> widely used and practical
### LoRA
- Original linear layer: y = Wx
- LoRA update: 
+ Freeze W, train: W' = W + delta W <- delta W = BA
+ A (r, d_in)
+ B (d_out, r)
- LoRA trains low-rank updates instead of the full weight matrix
- Parameter count:
+ One linear layer (d_out, d_in): Parameters d_out d_in
+ LoRA rd_in + rd_out = r(d_in + d_out)

In [ ]:
# LoRA with PEFT
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj'],
    lora_dropout=0.05,
    task_type='SEQ_CLS',
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## Efficiency: memory & computation
- Cost: 
+ Model parameters
+ Gradients
+ Optimizer states
+ Activations for backward
+ Attention matrices
+ Padding waste
### Memory budget
AdamW full finetuning:
- Weights
- Gradients
- First moment
- Second moment
- Activations
- NOTE: Optimizer states dominate in memory, PEFT + quantization reduce this burden
### Attention cost
Attention: (B, h, n, n)
- Memory: O(Bhn^2)
- Computation: O(dn^2)
- Reduce sequence length -> efficiency gain max
### Efficiency levers
Efficiency is a system-level design problem
- Dynamic padding & bucketing
- Mixed precision: FP16/BF16
- Gradient accumulation
- Gradient checkpointing
- PEFT/LoRA
- Quantization
- FlashAttention / fused kernels
- Distillation and smaller checkpoints
### Gradient accumulation
- Large batch size may not fit in GPU
- Accumulate gradients over multiple smaller mini-batches: B_effective = B_micro x N_accumulate
- Increase effective batch size without increasing peak activation memory per step

In [ ]:
accum_steps = 4
optimizer.zero_grad(set_to_none=True)

for step, batch in enumerate(train_dataloader):
    out = model(**batch)
    loss = out.loss / accum_steps
    loss.backward()

    if (step + 1) % accum_steps == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

### Gradient checkpointing
- Do not store all intermediate activations
- Recompute some activations during backward pass
- Trade off: lower memory >< more compute
- Useful: Large models, long sequences, limited GPU

In [ ]:
model.gradient_checkpointing_enable()

# Some model may need (decoder models)
model.config.use_cache = False

## Quantization and inference efficiency
### Quantization
Represent weights/activations with lower precision: FP32 -> FP16/BF/16 -> INT8 -> 4-bit
- Lower memory
- Faster inference
- Larger models fit on smaller GPUs
- May reduce quality, evaluate on the target task and language
### Finetuning with quantization
Use quantized base model + train small adapters: quantized model + LoRA adapters
- Idea of QLoRA-style workflow

In [ ]:
# Quantized loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype='bfloat16',
)
model = AutoModelForCausalLM.from_pretrained(
    ckpt,
    quantization_config=bnb_config,
    device_map='auto',
)

### KV cache
- Solve decoder-only generations repeats attention over past tokens
- Store previous keys and values: K<=t, V<=t
- Compute only new token Q/K/V
- Essential for efficient autoregressive inference
### Throughput vs latency
- Latency: Time for one request to complete
- Throughput: Number of tokens or samples processed per second
- Trade-off: large batching improves throughput but may increase latency
## Failure modes and debugging
### Common failure nodes
Most failures are contract failures, not architecture failures
- Tokenizer mismatch with checkpoint
- Padding tokens included in loss
- Wrong label shape
- Wrong causal mask direction
- Data leakage
- Overfitting from too large model on small dataset
- Misleading metric on imbalanced labels
### Debug checklist
- Print one tokenized sample
- Decode input_ids back to text
- Check attention_mask
- Print model input/output shapes
- Overfit one small batches
- Compare train vs validation loss
- Inspect errors, not only aggregate metrics
### Overfit one batch test
- Before full training, verify that model can overfit a tiny batch
+ If loss cannot decrease: bug in data, labels, head, loss, optimizer
+ If tiny batch overfits but validation fail: generalization/data problem
- One of fastest sanity check in deep learning
### Reproducibility checklist
- Fixed random seeds
- Save dataset split indices
- Save tokenizer name & version
- Save model ckpt name
- Log max length, padding strategy, preprocessing
- Log learning rate, schedule, batch size, and precision
- A finetuned result is not reproducible unless the full pipeline is recorded
## Deployment checklist
### Before deployment
- Freeze extract tokenizer and preprocessing
- Export model with correct config
- Benchmark latency and throughput on target hardware
- Test edge cases and long inputs
- Monitor distribution shift after deployment
- IMPORTANT ENGINEERING RULE: Do not deploy only the model file, deploy the whole inference pipeline
### Deployment optimization options
- Use smaller checkpoint
- Quantize weights
- Use batching
- Use KV cache (for generation)
- use ONNX/TensorRT/optimized serving stack when appropriate
- Cache repeated inputs or embeddings when possible
- Optimization should be measured on the real deployment workload

#